In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
"""Word-level tokenization, padding, and embedding lookup."""
embedding_layer = nn.Embedding(7, 3, padding_idx=0)

token2id = {
    "PAD": 0,
    "hello": 1,
    "world": 2,
    "darkness": 3,
    "my": 4,
    "old": 5,
    "friend": 6,
}

sentences = [
    "hello world",
    "hello darkness my old friend",
]

split_sentences = [s.split() for s in sentences]
max_len = max(len(s) for s in split_sentences)
padded = [s + ["PAD"] * (max_len - len(s)) for s in split_sentences]
tokenized = torch.tensor([[token2id[w] for w in s] for s in padded])

with torch.no_grad():
    embeds = embedding_layer(tokenized)

tokenized, embeds.shape

In [ ]:
class Tokenizer:
    """Character-level tokenizer with SOS, EOS, and UNK tokens."""

    def __init__(self):
        vocab = "abcdefghijklmnopqrstuvwxyz.!?:;,'-\" \n"
        self.char_to_id = {char: i for i, char in enumerate(vocab)}
        self.char_to_id["UNK"] = len(vocab)
        self.char_to_id["<sos>"] = len(vocab) + 1
        self.char_to_id["<eos>"] = len(vocab) + 2
        self.id_to_char = {i: char for char, i in self.char_to_id.items()}

    def tokenize(self, text):
        """Encode text as a list of token ids."""
        text = text.lower()
        output = [self.char_to_id["<sos>"]]
        for c in text:
            if c in self.char_to_id:
                output.append(self.char_to_id[c])
            else:
                output.append(self.char_to_id["UNK"])
        return output + [self.char_to_id["<eos>"]]

    def untokenize(self, ids):
        """Decode a list of token ids back to characters."""
        return [self.id_to_char[id_] for id_ in ids]


tokenizer = Tokenizer()
tokens = tokenizer.tokenize("hello world")
tokenizer.untokenize(tokens)

In [ ]:
class SelfAttention(nn.Module):
    """Single-head causal self-attention."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.W_Q = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.W_K = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.d_k = hidden_dim

    def forward(self, x):
        """Return attended values and attention weights for input (batch, seq, hidden)."""
        seq_len = x.size(1)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        mask = torch.triu(torch.ones(seq_len, seq_len), dtype=torch.bool, diagonal=0)
        causal_mask = torch.zeros(seq_len, seq_len).masked_fill(mask, float("inf"))

        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        scores = scores + causal_mask
        att = F.softmax(scores, dim=-1)
        return att @ V, att

In [ ]:
class TransformerLayer(nn.Module):
    """One transformer block: attention, residual connections, and feed-forward MLP."""

    def __init__(self, hidden_dim, expansion_factor):
        super().__init__()
        self.attention = SelfAttention(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * expansion_factor),
            nn.ReLU(),
            nn.Linear(hidden_dim * expansion_factor, hidden_dim),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        h, att = self.attention(x)
        h = self.norm1(h + x)
        z = self.mlp(h)
        return self.norm2(z + h), att


class Transformer(nn.Module):
    """Small decoder-style transformer with token and positional embeddings."""

    def __init__(self, d_model=8, n_vocab=10, expansion_factor=4, num_layers=3, max_length=50):
        super().__init__()
        self.positional_encoding = nn.Embedding(max_length, d_model)
        self.embeddings = nn.Embedding(n_vocab, d_model)
        self.layers = nn.ModuleList(
            [TransformerLayer(d_model, expansion_factor) for _ in range(num_layers)]
        )

    def forward(self, input_ids):
        pos = torch.arange(input_ids.shape[1], device=input_ids.device)
        x = self.embeddings(input_ids) + self.positional_encoding(pos)
        attns = []
        for layer in self.layers:
            x, attn = layer(x)
            attns.append(attn)
        return x @ self.embeddings.weight.T, attns